**IMPORTS NECESARIOS**

In [1]:
# Basic imports 
import os
import torch
import torch.nn as nn
from pathlib import Path

In [2]:
# Run only one time
# 1. Get the base path
root = Path.cwd().parents[2]
# 2. Set that path as cwd
os.chdir(root)
# 3. Debug checking
print(f'·Aspect of the cwd:({Path.cwd()})')

·Aspect of the cwd:(/home/carlo/cuarto/GPT2_MOE/src)


In [11]:
# FS imports 
from gpt2_moe.model import GPTMOE
from gpt2_moe.train_test.configs import ExperimentConfig
from gpt2_moe.train_test.utils import Utils

**NEEDED BASIC CLASSES**

In [21]:
# Basic Imports
import torch

# Ignored positions in the targets (prompt and padding)
IGNORE_INDEX = -100


def per_task(correct:torch.Tensor, total:torch.Tensor, names:list[str])->dict:
    """
        Function aimed to turn per task counters
        into a {task: ratio} dict and provide a general
        interclass mean value

        Params:
            correct(torch.Tensor):
                Hits per task index (n_tasks,)
            total(torch.Tensor):
                Evaluated items per task index (n_tasks,)
            names(list[str]):
                Task names ordered by index

        Returns:
            results(dict):
                {task_name: ratio, ..., 'mean': macro mean}
    """
    # Build the dict: exclude unexistent classes: ordered experts training approach
    results = {names[i]: (correct[i] / total[i]).item() for i in range(len(names)) if total[i] > 0 }
    # Compute the general mean
    results['mean'] = sum(results.values()) / len(results)
    return results


def split_example(input_ids:torch.Tensor, target_ids:torch.Tensor)->tuple[torch.Tensor, torch.Tensor]:
    """
        Function aimed to recover the prompt and the answer
        from a padded example

        Params:
            input_ids(torch.Tensor):
                Padded input ids (N,)
            target_ids(torch.Tensor):
                Padded target ids (N,) including the prompt masked

        Returns:
            prompt(torch.Tensor):
                Prompt ids, the last one predicts the first answer token
            answer(torch.Tensor):
                Answer ids without <|endoftext|>
    """
    # First non-masked index from target
    ans_pos = torch.where(target_ids != IGNORE_INDEX)[0][0]
    # Answer is prompt +  tab (1)
    prompt = input_ids[:ans_pos + 1]
    # Get answer without padding (-100)
    answer = target_ids[target_ids != IGNORE_INDEX][:-1]
    return prompt, answer


def greedy_answer(model, prompt:torch.Tensor, eos_id:int, max_new:int)->list[int]:
    """
        Function aimed to generate an answer with greedy
        decoding until <|endoftext|> or max_new tokens 
        are reached

        Params:
            model(GPTMOE):
                The model to generate with
            prompt(torch.Tensor):
                Prompt ids (,) already on the device
            eos_id(int):
                Id of <|endoftext|>
            max_new(int):
                Max amount of generated tokens

        Returns:
            out(list[int]):
                Lis of generated ids without <|endoftext|>
    """
    # Get the context length
    context_length = model.pos_emb.num_embeddings
    # Modify ids shape (N, 1)
    ids = prompt.unsqueeze(0)
    output = []
    #print('Aspect of the ids:', ids, ids.shape)
    # Generative task
    for _ in range(max_new):
        # Logits of the last position (|V|,)
        next_id = torch.argmax(model(ids[:, -context_length:])[0, -1])
        # Check if answer is finished
        if next_id.item() == eos_id:
            break
        # Compose the output chain
        output.append(next_id.item())
        # Update the context window for the model
        ids = torch.cat([ids, next_id.view(1, 1)], dim = 1)

    return output


**TOKEN LEVEL CE**

In [100]:
def ce_per_token(model:GPTMOE, loader, device:str, **kwargs)->dict[str:float]:
    """
        Function aimed to compute per token CE

        Params:
            model(GPTMOE):
                The model to evaluate
            loader(DataLoader):
                Loader over a TaskDataset
            device (str):
                The selected device

        Returns:
            results(dict):
                A dict composed by task:score items
    """
    # Get the dataset and task names
    ds = loader.dataset
    names = ds.task_names
    n = len(loader)

    # Define the loss function
    loss = nn.CrossEntropyLoss(
        ignore_index = -100,
        reduction = 'none'
    )

    # Define storages
    loss_per_task = torch.zeros(len(names), dtype=torch.float64)
    tokens_per_task = torch.zeros(len(names), dtype=torch.long)

    # Set model to eval
    model.eval()
    with torch.no_grad():
        # Iterate over batches
        for batch in loader:
            input = batch['input_ids'].to(device = device)
            target = batch['target_ids'].to(device = device)
            task = batch['task'].to(device = device)
            # Generate prediction (B, S, |V|)
            logits = model(input)
            batch_size, seq_len, vocab_size = logits.shape
            print('(ce_t) Prediction aspect:', logits.shape)
            print('(ce_t) Answer shape:', target.shape)
            print('(ce_t) Task shape:', task.shape)

            # Apply CE (B*S, |V|) -> get loss per token -> (B, S)
            tokens_loss = loss(logits.reshape(-1, logits.shape[-1]), target.reshape(-1)).reshape(batch_size, seq_len)
            print('(ce_t) Loss Shape', tokens_loss.shape)
            
            # Filter tokens
            valid_mask = target!=-100
            print('(ce_t) Aspect of the mask:', valid_mask.shape)
         
            # Filter tasks (task is a mask)
            for task_id, name in enumerate(names):
                # Get examples from batch which belong to task
                task_mask = task==task_id
                print(F'     (ce_t)({name}) Aspect of the task mask:', valid_mask.shape)

                # Apply (B, S) and (B, 1) -->(B,S )
                selected_mask = torch.bitwise_and(valid_mask, task_mask.unsqueeze(1))

                # Populate storage
                loss_per_task[task_id]+=(torch.sum(
                   selected_mask * tokens_loss).item()
                )
                tokens_per_task[task_id]+= selected_mask.sum().item()

                
    # Compute final values
    results = {}
    final_mean = []
    for task_id, task in enumerate(names):
        # Get mean per task
        # 1. Check task tokens
        task_tokens = tokens_per_task[task_id].item()
        if task_tokens == 0:
            results[task] = 0.0
            continue
        # 2. Get loss
        task_loss = loss_per_task[task_id].item()
        # Populate dict -> mean ce per task
        results[task] = task_loss/task_tokens 
        # 3. Final mean contribution
        final_mean.append(results[task])

    # Include the final mean
    results["mean"] = torch.mean(torch.tensor(final_mean)).item()

    return results



In [107]:
# Create dataloaded
dl,_,_ = Utils.get_loaders(tasks = ["add_sub", "spell"], batch_size = 32, context_length = 64)

In [109]:
# Define the params
device = 'cuda'
config = ExperimentConfig(name = 'test_1.yaml')
model = GPTMOE(config.get_model_params())
model.to(device = device)
# Call the function
results = ce_per_token(model = model, loader = dl, device = device)

(ce_t) Prediction aspect: torch.Size([32, 52, 50257])
(ce_t) Answer shape: torch.Size([32, 52])
(ce_t) Task shape: torch.Size([32])
(ce_t) Loss Shape torch.Size([32, 52])
(ce_t) Aspect of the mask: torch.Size([32, 52])
     (ce_t)(add_sub) Aspect of the task mask: torch.Size([32, 52])
     (ce_t)(spell) Aspect of the task mask: torch.Size([32, 52])
(ce_t) Prediction aspect: torch.Size([32, 52, 50257])
(ce_t) Answer shape: torch.Size([32, 52])
(ce_t) Task shape: torch.Size([32])
(ce_t) Loss Shape torch.Size([32, 52])
(ce_t) Aspect of the mask: torch.Size([32, 52])
     (ce_t)(add_sub) Aspect of the task mask: torch.Size([32, 52])
     (ce_t)(spell) Aspect of the task mask: torch.Size([32, 52])
(ce_t) Prediction aspect: torch.Size([32, 52, 50257])
(ce_t) Answer shape: torch.Size([32, 52])
(ce_t) Task shape: torch.Size([32])
(ce_t) Loss Shape torch.Size([32, 52])
(ce_t) Aspect of the mask: torch.Size([32, 52])
     (ce_t)(add_sub) Aspect of the task mask: torch.Size([32, 52])
     (ce_t)(